# Étude de Cas: FIFA World Cup 2026 pipeline development

This notebook ingests raw JSON files from the landing zone (Volumes) into a Bronze RAW Delta Table. This process will not perform:

- Neither flattening nor transformations in the data (This process will be delegated into Silver layer)
- Incremental and idempotent data to maintain a stable pipeline

Aside of this, in this notebook will be added a Bronze Iceberg table creation. By definition, this won't be executed due to the Free Databricks Edition usage.

In [0]:
%sql
-- Step 0: Schema Creation
CREATE SCHEMA IF NOT EXISTS api_pipeline_football;

In [0]:
%sql
-- Step 1: Bronze Table Creation
CREATE TABLE IF NOT EXISTS api_football_pipeline.bronze_fixtures_raw
USING DELTA;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6756768911189985>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '-- Step 0.1: Bronze Table Creation\nCREATE TABLE IF NOT EXISTS api_football_pipeline.bronze_fixtures_raw\nUSING DELTA;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, line, 

The error above shows the limitations of Free Edition, in real praxis we should create dedicated schemas to each layer. In this particular case, all tbales will be created in the volume's default schema

In [0]:
%sql
-- Step 1: Delta Table Creation
CREATE TABLE IF NOT EXISTS workspace.default.bronze_fixtures
USING DELTA;

In [0]:
%sql
-- Step 2: COPY INTO Delta from input folder
COPY INTO workspace.default.bronze_fixtures
FROM '/Volumes/workspace/default/api_football_pipeline/input/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
525,525,0


# Alternative approaches

In a data engineering perspective, the solution above is best suited for scalability and mainteinance, however, depending on business needs we could modify the primary conditions to handle dfferent situations in upper layers.

## Pyspark Dataframe creation through direct input -> Column Flattening -> Delta Table with fixed schema

In this approach we are using a pyspark dataframe, to read a stream from the input folder, if this approach had continous ingestion of data, we use `availableNow = True` for the stream to read everything once.

Then a column flattening process is taking place in the pyspark df to get all the fixtures' properties. Then a delta table is created with the same fixed schema finally writing the pyspark to the table with the `saveAsTable` option.

In [0]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import *

# Volume path
input_path = "/Volumes/workspace/default/api_football_pipeline/input"
bronze_table = "bronze_fixtures"

# JSON files read
df_raw = (
    spark.read
        .option("multiLine", True)
        .option("mode", "PERMISSIVE")   # default, pero lo dejamos explícito
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .json(f"{input_path}")
)
display(df_raw.limit(5))

fixture,goals,league,score,teams
"List(2023-07-02T01:30:00+00:00, 1023111, List(1688261400, 1688265000), M. Ortíz, List(90, null, Match Finished, FT), 1688261400, UTC, List(Houston, Texas, 19418, Shell Energy Stadium))","List(0, 0)","List(World, null, 22, https://media.api-sports.io/football/leagues/22.png, CONCACAF Gold Cup, Group Stage - 2, 2023, true)","List(List(null, null), List(0, 0), List(0, 0), List(null, null))","List(List(5529, https://media.api-sports.io/football/teams/5529.png, Canada, null), List(5161, https://media.api-sports.io/football/teams/5161.png, Guatemala, null))"
"List(2023-07-04T22:30:00+00:00, 1023115, List(1688509800, 1688513400), K. Herrera, List(90, null, Match Finished, FT), 1688509800, UTC, List(Houston, Texas, 19418, Shell Energy Stadium))","List(2, 4)","List(World, null, 22, https://media.api-sports.io/football/leagues/22.png, CONCACAF Gold Cup, Group Stage - 3, 2023, true)","List(List(null, null), List(2, 4), List(1, 2), List(null, null))","List(List(2388, https://media.api-sports.io/football/teams/2388.png, Cuba, false), List(5529, https://media.api-sports.io/football/teams/5529.png, Canada, true))"
"List(2023-06-27T23:00:00+00:00, 1036607, List(1687906800, 1687910400), R. Vazquez, List(90, null, Match Finished, FT), 1687906800, UTC, List(Toronto, Ontario, 312, BMO Field))","List(2, 2)","List(World, null, 22, https://media.api-sports.io/football/leagues/22.png, CONCACAF Gold Cup, Group Stage - 1, 2023, true)","List(List(null, null), List(2, 2), List(1, 0), List(null, null))","List(List(10983, https://media.api-sports.io/football/teams/10983.png, Guadeloupe, null), List(5529, https://media.api-sports.io/football/teams/5529.png, Canada, null))"
"List(2023-07-09T23:30:00+00:00, 1051776, List(1688945400, 1688949000), M. Ortíz, List(120, null, Match Finished, PEN), 1688945400, UTC, List(Cincinnati, Ohio, 19229, TQL Stadium))","List(2, 2)","List(World, null, 22, https://media.api-sports.io/football/leagues/22.png, CONCACAF Gold Cup, Quarter-finals, 2023, true)","List(List(1, 1), List(1, 1), List(0, 0), List(2, 3))","List(List(5529, https://media.api-sports.io/football/teams/5529.png, Canada, false), List(2384, https://media.api-sports.io/football/teams/2384.png, USA, true))"
"List(2023-10-13T10:35:00+00:00, 1114142, List(1697193300, 1697196900), A. King, List(90, null, Match Finished, FT), 1697193300, UTC, List(Niigata, 952, Denka Big Swan Stadium))","List(1, 4)","List(World, null, 10, https://media.api-sports.io/football/leagues/10.png, Friendlies, Friendlies 1, 2023, false)","List(List(null, null), List(1, 4), List(0, 3), List(null, null))","List(List(5529, https://media.api-sports.io/football/teams/5529.png, Canada, false), List(12, https://media.api-sports.io/football/teams/12.png, Japan, true))"


In [0]:
#Initioal Schema
df_raw.printSchema()

root
 |-- fixture: struct (nullable = true)
 |    |-- date: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- periods: struct (nullable = true)
 |    |    |-- first: long (nullable = true)
 |    |    |-- second: long (nullable = true)
 |    |-- referee: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- elapsed: long (nullable = true)
 |    |    |-- extra: long (nullable = true)
 |    |    |-- long: string (nullable = true)
 |    |    |-- short: string (nullable = true)
 |    |-- timestamp: long (nullable = true)
 |    |-- timezone: string (nullable = true)
 |    |-- venue: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- goals: struct (nullable = true)
 |    |-- away: long (nullable = true)
 |    |-- home: long (nullable = true)
 |-- league: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |--

In [0]:
# Corrupt files validation
df_raw.filter(col("_corrupt_record").isNotNull()).count()
#It failed, so the files were uploaded correctly

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5840342585835396>, line 2
      1 # Corrupt files validation
----> 2 df_raw.filter(col("_corrupt_record").isNotNull()).count()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1929     query = self._plan.to_proto(self._session.client)
-> 1930     table, schema, self._execution_info = self._session.client.to_table(
   1931         query, self._plan.observati

In [0]:
# First flattening
df_landing = df_raw.select(
    # fixture.*
    col("fixture.date").alias("fixture_date"),
    col("fixture.id").alias("fixture_id"),
    col("fixture.periods.first").alias("period_first"),
    col("fixture.periods.second").alias("period_second"),
    col("fixture.referee").alias("referee"),
    col("fixture.status.elapsed").alias("status_elapsed"),
    col("fixture.status.extra").alias("status_extra"),
    col("fixture.status.long").alias("status_long"),
    col("fixture.status.short").alias("status_short"),
    col("fixture.timestamp").alias("timestamp"),
    col("fixture.timezone").alias("timezone"),
    col("fixture.venue.city").alias("venue_city"),
    col("fixture.venue.id").alias("venue_id"),
    col("fixture.venue.name").alias("venue_name"),
# goals.*
    col("goals.home").alias("goals_home"),
    col("goals.away").alias("goals_away"),
# league.*
    col("league.country").alias("league_country"),
    col("league.flag").alias("league_flag"),
    col("league.id").alias("league_id"),
    col("league.logo").alias("league_logo"),
    col("league.name").alias("league_name"),
    col("league.round").alias("league_round"),
    col("league.season").alias("league_season"),
    col("league.standings").alias("league_standings"),
# score.*
    col("score.extratime.home").alias("extratime_home"),
    col("score.extratime.away").alias("extratime_away"),
    col("score.fulltime.home").alias("fulltime_home"),
    col("score.fulltime.away").alias("fulltime_away"),
    col("score.halftime.home").alias("halftime_home"),
    col("score.halftime.away").alias("halftime_away"),
    col("score.penalty.home").alias("penalty_home"),
    col("score.penalty.away").alias("penalty_away"),
# teams.*
    col("teams.home.id").alias("home_team_id"),
    col("teams.home.name").alias("home_team_name"),
    col("teams.home.logo").alias("home_team_logo"),
    col("teams.home.winner").alias("home_team_winner"),
col("teams.away.id").alias("away_team_id"),
    col("teams.away.name").alias("away_team_name"),
    col("teams.away.logo").alias("away_team_logo"),
    col("teams.away.winner").alias("away_team_winner")
)

In [0]:
display(df_landing.limit(5))

fixture_date,fixture_id,period_first,period_second,referee,status_elapsed,status_extra,status_long,status_short,timestamp,timezone,venue_city,venue_id,venue_name,goals_home,goals_away,league_country,league_flag,league_id,league_logo,league_name,league_round,league_season,league_standings,extratime_home,extratime_away,fulltime_home,fulltime_away,halftime_home,halftime_away,penalty_home,penalty_away,home_team_id,home_team_name,home_team_logo,home_team_winner,away_team_id,away_team_name,away_team_logo,away_team_winner
2023-07-02T01:30:00+00:00,1023111,1688261400,1688265000,M. Ortíz,90,null,Match Finished,FT,1688261400,UTC,"Houston, Texas",19418,Shell Energy Stadium,0,0,World,null,22,https://media.api-sports.io/football/leagues/22.png,CONCACAF Gold Cup,Group Stage - 2,2023,true,null,null,0,0,0,0,null,null,5161,Guatemala,https://media.api-sports.io/football/teams/5161.png,null,5529,Canada,https://media.api-sports.io/football/teams/5529.png,null
2023-07-04T22:30:00+00:00,1023115,1688509800,1688513400,K. Herrera,90,null,Match Finished,FT,1688509800,UTC,"Houston, Texas",19418,Shell Energy Stadium,4,2,World,null,22,https://media.api-sports.io/football/leagues/22.png,CONCACAF Gold Cup,Group Stage - 3,2023,true,null,null,4,2,2,1,null,null,5529,Canada,https://media.api-sports.io/football/teams/5529.png,true,2388,Cuba,https://media.api-sports.io/football/teams/2388.png,false
2023-06-27T23:00:00+00:00,1036607,1687906800,1687910400,R. Vazquez,90,null,Match Finished,FT,1687906800,UTC,"Toronto, Ontario",312,BMO Field,2,2,World,null,22,https://media.api-sports.io/football/leagues/22.png,CONCACAF Gold Cup,Group Stage - 1,2023,true,null,null,2,2,0,1,null,null,5529,Canada,https://media.api-sports.io/football/teams/5529.png,null,10983,Guadeloupe,https://media.api-sports.io/football/teams/10983.png,null
2023-07-09T23:30:00+00:00,1051776,1688945400,1688949000,M. Ortíz,120,null,Match Finished,PEN,1688945400,UTC,"Cincinnati, Ohio",19229,TQL Stadium,2,2,World,null,22,https://media.api-sports.io/football/leagues/22.png,CONCACAF Gold Cup,Quarter-finals,2023,true,1,1,1,1,0,0,3,2,2384,USA,https://media.api-sports.io/football/teams/2384.png,true,5529,Canada,https://media.api-sports.io/football/teams/5529.png,false
2023-10-13T10:35:00+00:00,1114142,1697193300,1697196900,A. King,90,null,Match Finished,FT,1697193300,UTC,Niigata,952,Denka Big Swan Stadium,4,1,World,null,10,https://media.api-sports.io/football/leagues/10.png,Friendlies,Friendlies 1,2023,false,null,null,4,1,3,0,null,null,12,Japan,https://media.api-sports.io/football/teams/12.png,true,5529,Canada,https://media.api-sports.io/football/teams/5529.png,false


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_fixtures_spark (
    fixture_date STRING,
    fixture_id BIGINT,
    period_first BIGINT,
    period_second BIGINT,
    referee STRING,
    status_elapsed BIGINT,
    status_extra BIGINT,
    status_long STRING,
    status_short STRING,
    timestamp BIGINT,
    timezone STRING,
    venue_city STRING,
    venue_id BIGINT,
    venue_name STRING,

    goals_home BIGINT,
    goals_away BIGINT,

    league_country STRING,
    league_flag STRING,
    league_id BIGINT,
    league_logo STRING,
    league_name STRING,
    league_round STRING,
    league_season BIGINT,
    league_standings BOOLEAN,

    extratime_home BIGINT,
    extratime_away BIGINT,
    fulltime_home BIGINT,
    fulltime_away BIGINT,
    halftime_home BIGINT,
    halftime_away BIGINT,
    penalty_home BIGINT,
    penalty_away BIGINT,

    home_team_id BIGINT,
    home_team_name STRING,
    home_team_logo STRING,
    home_team_winner BOOLEAN,

    away_team_id BIGINT,
    away_team_name STRING,
    away_team_logo STRING,
    away_team_winner BOOLEAN
)
USING DELTA;


The following command is not incremental, is to append snapshots of the data, therfore there will be duplicated in bronze

In [0]:
df_landing.write.format("delta").mode("append").saveAsTable("bronze_fixtures_spark")

For it to be incremental it is needed the Delta Lake library to perform an upsert with `merge`

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "workspace.default.bronze_fixture_spark")

(
    deltaTable.alias("t")
    .merge(
        df_landing.alias("s"),
        "t.id = s.id"   # condición de emparejamiento
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

## Pyspark DataFrame -> Incremental Iceberg

In this case either df_raw or df_landing (the flat dataframe) can be transported to an Iceberg table using a variation of the `CREATE TABLE IF NOT EXISTS` command, just replacing the `USING DELTA` to `USING ICEBERG`

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_fixtures_iceberg (
    fixture_date STRING,
    fixture_id BIGINT,
    period_first BIGINT,
    period_second BIGINT,
    referee STRING,
    status_elapsed BIGINT,
    status_extra BIGINT,
    status_long STRING,
    status_short STRING,
    timestamp BIGINT,
    timezone STRING,
    venue_city STRING,
    venue_id BIGINT,
    venue_name STRING,

    goals_home BIGINT,
    goals_away BIGINT,

    league_country STRING,
    league_flag STRING,
    league_id BIGINT,
    league_logo STRING,
    league_name STRING,
    league_round STRING,
    league_season BIGINT,
    league_standings BOOLEAN,

    extratime_home BIGINT,
    extratime_away BIGINT,
    fulltime_home BIGINT,
    fulltime_away BIGINT,
    halftime_home BIGINT,
    halftime_away BIGINT,
    penalty_home BIGINT,
    penalty_away BIGINT,

    home_team_id BIGINT,
    home_team_name STRING,
    home_team_logo STRING,
    home_team_winner BOOLEAN,

    away_team_id BIGINT,
    away_team_name STRING,
    away_team_logo STRING,
    away_team_winner BOOLEAN
)
USING ICEBERG;


Then to make an incremental insertion we use the `MERGE` command

In [0]:
%sql
MERGE INTO workspace.default.bronze_fixtures_iceberg t
USING df_incremental s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;